In [1]:
pip install choix numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

import pandas as pd
import json
import ast 

import numpy as np
import choix


In [3]:
def load_dicts_to_df(
    directory: str | Path,
    pattern: str = "*",
) -> pd.DataFrame:
    directory = Path(directory)
    if not directory.is_dir():
        raise NotADirectoryError(directory)

    rows: list[dict[str, Any]] = []
    for path in sorted(directory.glob(pattern)):
        if not path.is_file() or path.name.startswith("."):
            continue
        suffix = path.suffix.lower()
        data = None
        if suffix == ".json":
            with path.open(encoding="utf-8") as f:
                data = json.load(f)
        else:
            text = path.read_text(encoding="utf-8")
            data = ast.literal_eval(text)

        if not isinstance(data, dict):
            raise TypeError(f"{path} does not contain a dict (got {type(data).__name__})")
        rows.append(data)

    return pd.json_normalize(rows)



In [4]:
# git clone https://huggingface.co/spaces/taagarwa/coding-agent-leaderboard
df = load_dicts_to_df("/home/eje/git/coding-agent-leaderboard/results", pattern="*.json")

In [5]:
df.columns

Index(['benchmark.name', 'benchmark.repo', 'benchmark.num_tasks',
       'benchmark.url', 'harness.name', 'harness.skills', 'harness.is_oss',
       'harness.url', 'model.name', 'model.repo', 'model.is_oss',
       'model.num_params', 'model.precision', 'model.url', 'environment.name',
       'environment.config.path', 'environment.config.name',
       'environment.config.version', 'environment.config.ref',
       'environment.config.registry_url', 'environment.config.registry_path',
       'environment.config.overwrite', 'environment.config.download_dir',
       'environment.config.task_names',
       'environment.config.exclude_task_names', 'environment.config.n_tasks',
       'environment.url', 'metrics.n_tasks', 'metrics.n_errors',
       'metrics.score', 'metrics.n_input_tokens', 'metrics.n_cache_tokens',
       'metrics.n_output_tokens', 'metrics.n_total_tokens',
       'metrics.agent_time_seconds', 'metrics.total_time_seconds',
       'metrics.cost_usd', 'metrics.mean_input_toke

In [6]:
from typing import Any
def prepare_ranking_data(df: pd.DataFrame,
                         catcol: str | list[str],
                         metcol: str,
                         descending: bool = False) -> tuple[list[tuple[int, int]], list[Any]]:
    ndata = df.shape[0]
    if ndata < 2:
        raise ValueError("Not enough data to prepare ranking comparisons")
    catcol = catcol if isinstance(catcol, list) else [catcol]
    ncat = len(catcol)
    tcols = catcol + [metcol]
    t = list(df[tcols].itertuples(index=False, name=None))
    metvals = [x[-1] for x in t]
    if ncat > 1:
        catvals = [x[:-1] for x in t]
    else:
        # single category column: categories are just the values of the column
        catvals = [x[0] for x in t]
    # map item categories to unique integers
    umap = dict([(y,x) for x,y in enumerate(sorted(set(catvals)))])
    compvals = [(umap[c],m) for c,m in zip(catvals, metvals)]
    comps = []
    for i in range(ndata):
        ic, im = compvals[i]
        for j in range(i):
            jc, jm = compvals[j]
            # ignore items with same metric value
            if im == jm: continue
            # pairs are always of form (winner, loser)
            if im > jm:
                comps.append((jc, ic) if descending else (ic, jc))
            else:
                comps.append((ic, jc) if descending else (jc, ic))
    return comps, sorted(umap.keys())


In [7]:
comps, cats = prepare_ranking_data(df, 'harness.name', 'metrics.score')
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank}. {params[idx]:+.3f}  {cats[idx]}")

Ranking:
  1. +1.025  Codex
  2. +0.518  Claude Code
  3. -0.113  OpenCode
  4. -0.191  Qwen Code
  5. -0.530  Pi
  6. -0.709  OpenClaw


In [8]:
comps, cats = prepare_ranking_data(df, 'model.name', 'metrics.score')
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank}. {params[idx]:+.3f}  {cats[idx]}")


Ranking:
  1. +1.951  Opus 4.8
  2. +0.499  GPT 5.5 - high
  3. -0.123  Opus 4.6
  4. -0.581  Sonnet 4.6
  5. -1.745  Qwen3.6-35B-A3B-NVFP4


In [9]:
comps, cats = prepare_ranking_data(df, ['model.name', 'harness.name'], 'metrics.score')
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank}. {params[idx]:+.3f}  {cats[idx]}")


Ranking:
  1. +2.472  ('Opus 4.8', 'Claude Code')
  2. +2.472  ('Opus 4.8', 'OpenCode')
  3. +1.028  ('GPT 5.5 - high', 'Codex')
  4. +0.399  ('Opus 4.6', 'Claude Code')
  5. -0.069  ('Sonnet 4.6', 'Claude Code')
  6. -0.683  ('Qwen3.6-35B-A3B-NVFP4', 'Qwen Code')
  7. -0.812  ('Qwen3.6-35B-A3B-NVFP4', 'Claude Code')
  8. -1.119  ('Qwen3.6-35B-A3B-NVFP4', 'Pi')
  9. -1.345  ('Qwen3.6-35B-A3B-NVFP4', 'OpenClaw')
  10. -2.342  ('Qwen3.6-35B-A3B-NVFP4', 'OpenCode')


In [10]:
comps, cats = prepare_ranking_data(df, ['model.name', 'harness.name'], 'metrics.mean_cost_usd_per_task', descending=True)
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank}. {params[idx]:+.3f}  {cats[idx]}")


Ranking:
  1. +2.107  ('Qwen3.6-35B-A3B-NVFP4', 'Qwen Code')
  2. +2.107  ('Qwen3.6-35B-A3B-NVFP4', 'Claude Code')
  3. +2.107  ('Qwen3.6-35B-A3B-NVFP4', 'OpenClaw')
  4. +1.422  ('Qwen3.6-35B-A3B-NVFP4', 'OpenCode')
  5. +1.014  ('Qwen3.6-35B-A3B-NVFP4', 'Pi')
  6. -1.069  ('Sonnet 4.6', 'Claude Code')
  7. -1.468  ('Opus 4.8', 'OpenCode')
  8. -2.007  ('Opus 4.6', 'Claude Code')
  9. -2.009  ('Opus 4.8', 'Claude Code')
  10. -2.204  ('GPT 5.5 - high', 'Codex')
